# Singh manuscript evidence verification

This portable notebook checks the shipped frozen evidence used by manuscript v5. It does not refit the WT partition, Palantir coordinates/masks, or NMF, and does not turn conditional within-experiment results into independent biological validation. Current source notebooks are preserved with publication path aliases in `source_notebooks/`.

Original local execution environment: existing chronoclone installation, `/path/to/ChronoClone/.runtime/chronoclone`, kernel `chronoclone-paper`. Run from this notebook's directory: `/path/to/ChronoClone/.runtime/chronoclone/bin/python -m jupyter nbconvert --execute --to notebook --inplace --ExecutePreprocessor.kernel_name=chronoclone-paper 0001_validate_published_evidence.ipynb`. Other installations may use the pinned package versions documented in the source environment records; this verification itself needs Python, NumPy, pandas, SciPy, statsmodels and nbformat.


Publication verification command, run from this directory: `JUPYTER_PATH=/path/to/ChronoClone/.runtime/jupyter /path/to/ChronoClone/.runtime/chronoclone/bin/python -m jupyter nbconvert --execute --to notebook --inplace --ExecutePreprocessor.kernel_name=chronoclone-paper 0001_validate_published_evidence.ipynb`.

In [1]:
from pathlib import Path
import sys, json, hashlib, platform, importlib.metadata as metadata
import numpy as np, pandas as pd, nbformat
from statsmodels.stats.multitest import multipletests
from IPython.display import display
HERE=Path.cwd()
assert (HERE/'provenance/copy_plan.json').is_file(), 'Run with working directory analysis/singh'
environment={'name':'chronoclone','manager':'existing Conda-compatible installation','local_recorded_prefix':'/path/to/ChronoClone/.runtime/chronoclone','actual_prefix':str(Path(sys.executable).parent.parent),'kernel':'chronoclone-paper','Python':platform.python_version(),'versions':{p:metadata.version(p) for p in ['numpy','pandas','scipy','statsmodels','nbformat','nbclient']},'command':'python -m jupyter nbconvert --execute --to notebook --inplace --ExecutePreprocessor.kernel_name=chronoclone-paper 0001_validate_published_evidence.ipynb'}
display(environment)
checks=[]
def check(name,condition):
    checks.append({'check':name,'passed':bool(condition)})
    assert condition,name
def sha(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()
manifest=json.loads((HERE/'provenance/copy_plan.json').read_text())['files']
for r in manifest:check('hash '+r['destination'],sha(HERE/r['destination'])==r['sha256'])
print('Verified copied artifacts:',len(manifest))


{'name': 'chronoclone',
 'manager': 'existing Conda-compatible installation',
 'local_recorded_prefix': '/path/to/ChronoClone/.runtime/chronoclone',
 'actual_prefix': '/path/to/ChronoClone/.runtime/chronoclone',
 'kernel': 'chronoclone-paper',
 'Python': '3.11.15',
 'versions': {'numpy': '2.4.6',
  'pandas': '2.3.3',
  'scipy': '1.17.1',
  'statsmodels': '0.14.6',
  'nbformat': '5.10.4',
  'nbclient': '0.11.0'},
 'command': 'python -m jupyter nbconvert --execute --to notebook --inplace --ExecutePreprocessor.kernel_name=chronoclone-paper 0001_validate_published_evidence.ipynb'}

Verified copied artifacts: 348


## Resolved families and planned comparisons

In [2]:
D=pd.read_csv(HERE/'tables/dynamics/cells_resolved.csv',dtype={'mc':str,'clone':str})
check('resolved cohort',len(D)==14896 and D.cell_id.is_unique and D.clone.nunique()==1516)
check('one metaclone per family',D.groupby('clone').mc.nunique().eq(1).all())
check('day7 cohort',D.loc[D.day.eq('D7')].shape[0]==2713 and D.loc[D.day.eq('D7'),'clone'].nunique()==1514)
check('HSPC pooling',D.loc[D.day.eq('D7'),'ct'].eq('HSPC').all())
E=pd.read_csv(HERE/'tables/dynamics/effects.csv')
check('68 planned comparisons',len(E)==68 and not E.duplicated(['mc','day','mutant']).any())
for endpoint in ['state','composition']:
    p=E['p_'+endpoint];q=multipletests(p.fillna(1),method='fdr_bh')[1]
    check(endpoint+' BH',np.allclose(q[p.notna()],E.loc[p.notna(),'q_'+endpoint]))
L=pd.read_csv(HERE/'tables/lineage_responses/lineage_state_effects.csv')
check('340 planned lineage comparisons',len(L)==340 and not L.duplicated(['mc','day','mutant','branch']).any())
p=L.p;q=multipletests(p.fillna(1),method='fdr_bh')[1]
check('global lineage BH',np.allclose(q[p.notna()],L.loc[p.notna(),'q_340']))
display(pd.DataFrame([{'cohort_cells':len(D),'families':D.clone.nunique(),'metaclones':D.mc.nunique(),'state_q05':int((E.q_state<.05).sum()),'composition_q05':int((E.q_composition<.05).sum()),'lineage_q05':int((L.q_340<.05).sum())}]))


,cohort_cells,families,metaclones,state_q05,composition_q05,lineage_q05
0,14896,1516,17,12,17,29


## Early RNA program evidence and atlas coverage

In [3]:
U=pd.read_csv(HERE/'tables/programs/day7_family_program_usage.csv',dtype={'mc':str,'clone':str})
cols=['P'+str(i) for i in range(1,9)]
check('program cohort',len(U)==1514 and U.clone.is_unique)
check('unit-sum usage',np.allclose(U[cols].sum(axis=1),1))
means=U.groupby('mc')[cols].mean()
stored=pd.read_csv(HERE/'tables/programs/metaclone_mean_usage.csv',index_col=0)
stored.index=stored.index.astype(str)
check('family-balanced program means',np.allclose(means.sort_index(),stored[cols].sort_index()))
enrich=pd.read_csv(HERE/'tables/programs/program_enrichment_complete_family.csv')
check('complete enrichment family',len(enrich)==30624)
atlas=list((HERE/'atlas/svg').glob('mc*.svg'))
check('all individual atlas maps',len(atlas)==119)
check('editable atlas XML',all('<svg' in p.read_text() for p in atlas))
display(means.loc[['3','7']].mul(100).round(3))
print('Complete enrichment rows:',len(enrich),'individual maps:',len(atlas))


,P1,P2,P3,P4,P5,P6,P7,P8
mc,,,,,,,,
3,21.644,22.357,9.266,10.949,3.553,8.123,10.713,13.394
7,12.371,17.701,8.744,16.015,17.656,7.808,14.700,5.005


Complete enrichment rows: 30624 individual maps: 119


## Preserved execution and verification receipt

In [4]:
records=[]
for r in manifest:
    if r['role']!='executed_source_notebook':continue
    p=HERE/r['destination'];b=nbformat.read(p,as_version=4)
    code_cells=[c for c in b.cells if c.cell_type=='code' and c.source.strip()]
    errors=[o for c in code_cells for o in c.get('outputs',[]) if o.get('output_type')=='error']
    records.append({'notebook':r['destination'],'code_cells':len(code_cells),'executed_cells':sum(c.get('execution_count') is not None for c in code_cells),'stored_errors':len(errors)})
    check('no stored errors '+p.name,not errors)
display(pd.DataFrame(records))
receipt={'status':'passed','check_count':len(checks),'source_artifact_count':len(manifest),'environment':environment,'scope':'Frozen evidence integrity and selected numerical reconciliation; no source scientific notebook rerun or original partition refit','source_notebooks':records}
(HERE/'provenance/evidence_validation.json').write_text(json.dumps(receipt,indent=2)+'\n')
print('Passed',len(checks),'checks. Original 17-metaclone fit and Palantir construction remain upstream provenance gaps.')


,notebook,code_cells,executed_cells,stored_errors
0,source_notebooks/dynamics/0001_family_aware_re...,11,11,0
1,source_notebooks/dynamics/0002_mc3_branch_resp...,7,7,0
2,source_notebooks/dynamics/0003_core_panel_visu...,2,2,0
3,source_notebooks/programs/0002_rna_normalized_...,9,9,0
4,source_notebooks/lineage_responses/0001_lineag...,14,14,0
5,source_notebooks/lineage_responses/0002_execut...,1,1,0
6,source_notebooks/program_annotations/0001_gene...,15,15,0
7,source_notebooks/program_annotations/0002_anno...,1,1,0
8,source_notebooks/landscape/0001_reference_stor...,4,4,0
9,source_notebooks/landscape/0001_singh_landscap...,8,8,0


Passed 377 checks. Original 17-metaclone fit and Palantir construction remain upstream provenance gaps.
